# Industry Grade Project 1 — Visual DevOps to DevSecOps Runbook

**Source fidelity:** The original Word document remains preserved as the authoritative source for the Jenkins screens, Maven logs, Tomcat deployment, Docker execution, Ansible automation, and Kubernetes evidence. This notebook organizes those steps and adds a clearly labeled AWS observability and security extension.

**Tags:** `jenkins` `maven` `tomcat` `docker` `ansible` `ecr` `kubernetes` `prometheus` `grafana` `devsecops`


## Visual architecture

![Visual DevSecOps flow](../docs/visuals/devsecops-flow.svg)

This visual replaces the earlier arrow-only Mermaid block. It shows the complete path from GitHub and Jenkins through Docker, Ansible, Amazon ECR, Kubernetes replicas, the Service/load balancer, ADOT, Amazon Managed Service for Prometheus, Amazon Managed Grafana, and SNS/SIEM response.


## 1. Jenkins and Maven build

The original exercise configures Jenkins to pull the repository, execute Maven `clean install`, run tests, package `webapp.war`, archive artifacts, and deploy the WAR to Tomcat.


In [ ]:
mvn clean install


**Completion evidence:** successful Git checkout, two passing server tests, WAR packaging, archived JAR/WAR/POM artifacts, Tomcat redeployment, and `Finished: SUCCESS`. The Word document retains the exact screenshots and full console output.


## 2. Standalone Docker deployment

The Docker stage is shown before Kubernetes, matching the original exercise sequence. Jenkins transfers the WAR to the Docker host, builds the Tomcat image, removes the previous container, and starts the new container.


In [ ]:
cd /opt/docker
docker build -t regappv1 .
docker stop registerapp || true
docker rm registerapp || true
docker run -d --name registerapp -p 8087:8080 regappv1
docker ps


In [ ]:
FROM tomcat:latest
RUN cp -R /usr/local/tomcat/webapps.dist/* /usr/local/tomcat/webapps
COPY ./*.war /usr/local/tomcat/webapps


Modernization note: pin the Tomcat image version or digest, run as a non-root user, remove unused default applications, and add a health check.


## 3. Ansible automation

The exercise transfers `webapp.war` to the Ansible server and runs a playbook that builds, tags, and pushes the image.


In [ ]:
---
- hosts: ansible
  tasks:
    - name: create docker image
      command: docker build -t regapp:latest .
      args:
        chdir: /opt/docker
    - name: tag image
      command: docker tag regapp:latest <registry-user>/regapp:latest
    - name: push image
      command: docker push <registry-user>/regapp:latest


In [ ]:
docker login
ansible-playbook /opt/docker/create_image_regapp.yml
sleep 10


## 4. Amazon ECR extension

The historical exercise used a public container registry. The AWS equivalent uses short-lived IAM credentials and ECR.


In [ ]:
aws ecr get-login-password --region <region> | docker login --username AWS --password-stdin <account>.dkr.ecr.<region>.amazonaws.com
docker tag regapp:latest <account>.dkr.ecr.<region>.amazonaws.com/regapp:<version>
docker push <account>.dkr.ecr.<region>.amazonaws.com/regapp:<version>


## 5. Kubernetes deployment

The Deployment manages a ReplicaSet, which maintains three application Pod replicas. A Service selects healthy Pods and provides stable routing. An Ingress or `LoadBalancer` Service provides the external URL.


In [ ]:
kubectl apply -f k8s/deployment.yaml
kubectl apply -f k8s/service.yaml
kubectl rollout status deployment/retail-portal
kubectl get deployment,rs,pods,svc -o wide


## 6. AWS metrics and dashboard configuration

Follow the complete configuration guide in [`docs/AWS_PROMETHEUS_RUNBOOK.md`](../docs/AWS_PROMETHEUS_RUNBOOK.md). The runbook covers AMP workspace creation, IAM through EKS workload identity or IRSA, ADOT scraping, SigV4 remote write, AMG data-source configuration, PromQL, alert rules, SNS routing, log forwarding, privacy controls, and validation.

### Dashboard mockup

![Illustrative Amazon Managed Grafana dashboard](../docs/visuals/aws-prometheus-dashboard.svg)

The mock dashboard is intentionally labeled illustrative. It visualizes replicas, request rate, p95 latency, 5xx rate, CPU, memory, and alert state. Replace synthetic values with real AMP and CloudWatch data.


## 7. PromQL examples


In [ ]:
# Available replicas
kube_deployment_status_replicas_available{deployment="retail-portal"}

# p95 latency
histogram_quantile(0.95, sum(rate(http_server_requests_seconds_bucket[5m])) by (le))

# 5xx ratio
sum(rate(http_server_requests_seconds_count{status=~"5.."}[5m])) / sum(rate(http_server_requests_seconds_count[5m]))


## 8. Enterprise security add-on

The security stage is an extension, not represented as original lab evidence. Recommended order: Contrast test instrumentation → Veracode SAST → Nexus IQ dependency policy → Docker build → Aqua image scan → ECR promotion → Kubernetes test deployment → Burp Suite DAST → promote or block.

- **Veracode:** static analysis and policy evaluation
- **Nexus IQ:** dependency, CVE, and license policy
- **Contrast:** IAST during integration tests
- **Aqua:** image and runtime posture analysis
- **Burp Suite:** DAST against the test endpoint


## 9. Evidence and privacy handling

The original Word document remains in the repository. Public notebooks should redact personal names where unnecessary, private IPs, hostnames, account IDs, tokens, credentials, internal URLs, and sensitive screenshots. Metrics must never use PII as labels.
